# Lastnost optimalne podstrukture

## Hitro potenciranje

$$\begin{aligned}
  a^0 &= 1 \\
  a^n &= a \cdot a^{n - 1}
\end{aligned}$$

In [ ]:
let rec potenciraj a =
    function
    | 0 -> 1
    | n -> a * potenciraj a (n - 1)

In [ ]:
pocasi_potenciraj 2 10

In [ ]:
let hitri_fib =
    let rec aux a b = function
    | 0 -> a
    | n -> aux (b mod ostanek) ((a + b) mod ostanek) (n - 1)
    in
    aux 0 1

In [ ]:
type matrika = {
      _11: int; _12: int; _21: int; _22: int
  }

In [ ]:
let zmnozi a b = {
      _11 = a._11 * b._11 + a._12 * b._21;
      _21 = a._21 * b._11 + a._22 * b._21;
      _12 = a._11 * b._12 + a._12 * b._22;
      _22 = a._21 * b._12 + a._22 * b._22
  }

In [ ]:
let ostanek_matrike a m = {
      _11 = a._11 mod m;
      _12 = a._12 mod m;
      _21 = a._21 mod m;
      _22 = a._22 mod m
  }

In [ ]:
let rec potenciraj_z zmnozi enota a =
    function
    | 0 -> enota
    | n ->
        let k = potenciraj_z zmnozi enota a (n / 2) in
        if n mod 2 = 0 then zmnozi k k else zmnozi a (zmnozi k k)

In [ ]:
potenciraj_z ( * ) 1 2 10

In [ ]:
let stopaj f x =
    let zacetek = Sys.time () in
    let y = f x in
    let konec = Sys.time () in
    let izpis =
      Printf.sprintf "Porabljen čas: %f ms" (1000. *. (konec -. zacetek))
    in
    print_endline izpis;
    y

## Stolpi iz kock

<div class="diagram" ondblclick="event.stopPropagation()">
<div class="diagram-step"></div>
</div>

In [ ]:
let rec stolpi =
    ...

In [ ]:
stolpi 20

In [ ]:
List.init 10 stolpi

In [ ]:
[22; 24; 26; 28]
  |> List.iter (stopaj (fun n -> ignore @@ stolpi n))

In [ ]:
let hitri_stolpi n =
    let rec aux n a b c =
      if n = 0 then c else aux (n - 1) b c (a + b + c)
    in
    aux n 0 0 1

In [ ]:
List.init 10 hitri_stolpi

In [ ]:
hitri_stolpi 50

## Najcenejša pot v matriki

<div class="diagram" ondblclick="event.stopPropagation()">
<div class="diagram-step"></div>
<div class="diagram-step" hidden></div>
<div class="diagram-step" hidden></div>
<div class="diagram-nav"><button type="button" data-d="-1" aria-label="nazaj" onclick="var f = this.closest('.diagram'), s = f.querySelectorAll('.diagram-step'), i = f.querySelector('.diagram-index'), k = i.textContent - 1; s[k].hidden = true; k = (k + s.length + Number(this.dataset.d)) % s.length; s[k].hidden = false; i.textContent = k + 1">&lsaquo;</button> <span class="diagram-index">1</span>/3 <button type="button" data-d="1" aria-label="naprej" onclick="var f = this.closest('.diagram'), s = f.querySelectorAll('.diagram-step'), i = f.querySelector('.diagram-index'), k = i.textContent - 1; s[k].hidden = true; k = (k + s.length + Number(this.dataset.d)) % s.length; s[k].hidden = false; i.textContent = k + 1">&rsaquo;</button></div>
</div>

In [ ]:
let matrika = [|
    [|131; 673; 234; 103; 18|];
    [|201; 96; 342; 965; 150|];
    [|630; 803; 746; 422; 111|];
    [|537; 699; 497; 121; 956|];
    [|805; 732; 524; 37; 331|]
  |]

In [ ]:
let cena_najcenejse_poti matrika =
    let m = Array.length matrika and n = Array.length matrika.(0) in
    let rec cena_iz i j =
      let polje = matrika.(i).(j) in
      match i = m - 1, j = n - 1 with
      | true, true -> polje
      | true, false -> polje + cena_iz i (j + 1)
      | false, true -> polje + cena_iz (i + 1) j
      | false, false -> polje + min (cena_iz (i + 1) j) (cena_iz i (j + 1))
    in
    cena_iz 0 0

In [ ]:
cena_najcenejse_poti matrika

In [ ]:
let nakljucna_matrika n =
    Array.init n (fun _ -> Array.init n (fun _ -> Random.int 1000))

In [ ]:
[10; 11; 12; 13]
  |> List.map nakljucna_matrika
  |> List.iter (stopaj (fun mat -> ignore @@ cena_najcenejse_poti mat))

In [ ]:
najcenejsa_pot matrika

## Krčenje slik

`slike.ml`

In [ ]:
let razsiri_lzw najmanjsa_velikost podatki stevilo_pikslov =
  let izhod = Array.make stevilo_pikslov 0 and zapisanih = ref 0 in
  let polozaj = ref 0 in
  let preberi velikost =
    let koda = ref 0 in
    for k = 0 to velikost - 1 do
      let bit = !polozaj + k in
      if Char.code podatki.[bit / 8] land (1 lsl (bit mod 8)) <> 0 then
        koda := !koda lor (1 lsl k)
    done;
    polozaj := !polozaj + velikost;
    !koda
  in
  let pocisti = 1 lsl najmanjsa_velikost in
  let konec = pocisti + 1 in
  let slovar = Array.init 4096 (fun i -> [| i |]) in
  let naslednja = ref (konec + 1) and velikost = ref (najmanjsa_velikost + 1) in
  let prejsnji = ref [||] in
  let nadaljuj = ref true in
  while !nadaljuj && !zapisanih < stevilo_pikslov do
    let koda = preberi !velikost in
    if koda = pocisti then (
      naslednja := konec + 1;
      velikost := najmanjsa_velikost + 1;
      prejsnji := [||])
    else if koda = konec then nadaljuj := false
    else
      let zaporedje =
        if koda < !naslednja then slovar.(koda)
        else Array.append !prejsnji [| !prejsnji.(0) |]
      in
      Array.iter
        (fun i ->
          if !zapisanih < stevilo_pikslov then (
            izhod.(!zapisanih) <- i;
            incr zapisanih))
        zaporedje;
      if !prejsnji <> [||] && !naslednja < 4096 then (
        slovar.(!naslednja) <- Array.append !prejsnji [| zaporedje.(0) |];
        incr naslednja;
        if !naslednja = 1 lsl !velikost && !velikost < 12 then incr velikost);
      prejsnji := zaporedje
  done;
  izhod

let preberi_gif ime_datoteke =
  let vsebina = In_channel.with_open_bin ime_datoteke In_channel.input_all in
  let bajt k = Char.code vsebina.[k] in
  let dva_bajta k = bajt k lor (bajt (k + 1) lsl 8) in
  let paleta zacetek velikost =
    Array.init velikost (fun i ->
        let k = zacetek + (3 * i) in
        (bajt k, bajt (k + 1), bajt (k + 2)))
  in
  let rec preskoci_bloke k = if bajt k = 0 then k + 1 else preskoci_bloke (k + 1 + bajt k) in
  let rec zberi_bloke k bloki =
    if bajt k = 0 then String.concat "" (List.rev bloki)
    else zberi_bloke (k + 1 + bajt k) (String.sub vsebina (k + 1) (bajt k) :: bloki)
  in
  let zastavice = bajt 10 in
  let globalna = paleta 13 (2 lsl (zastavice land 7)) in
  let rec slika k =
    match bajt k with
    | 0x21 -> slika (preskoci_bloke (k + 2))
    | 0x2C ->
        let sirina = dva_bajta (k + 5) and visina = dva_bajta (k + 7) in
        let zastavice = bajt (k + 9) in
        if zastavice land 0x40 <> 0 then failwith "preberi_gif: prepletena slika";
        let barve, k =
          if zastavice land 0x80 = 0 then (globalna, k + 10)
          else
            let velikost = 2 lsl (zastavice land 7) in
            (paleta (k + 10) velikost, k + 10 + (3 * velikost))
        in
        let indeksi = razsiri_lzw (bajt k) (zberi_bloke (k + 1) []) (sirina * visina) in
        Array.init visina (fun i -> Array.init sirina (fun j -> barve.(indeksi.((i * sirina) + j))))
    | _ -> failwith "preberi_gif: datoteka nima slike"
  in
  slika (if zastavice land 0x80 = 0 then 13 else 13 + (3 * Array.length globalna))

let lzw indeksi =
  let izhod = Buffer.create 4096 in
  let zbiralnik = ref 0 and stevilo_bitov = ref 0 in
  let zapisi koda velikost =
    zbiralnik := !zbiralnik lor (koda lsl !stevilo_bitov);
    stevilo_bitov := !stevilo_bitov + velikost;
    while !stevilo_bitov >= 8 do
      Buffer.add_char izhod (Char.chr (!zbiralnik land 0xFF));
      zbiralnik := !zbiralnik lsr 8;
      stevilo_bitov := !stevilo_bitov - 8
    done
  in
  let pocisti = 256 and konec = 257 in
  let slovar = Hashtbl.create 4096 in
  let naslednja = ref 258 and velikost = ref 9 in
  zapisi pocisti !velikost;
  let predpona = ref indeksi.(0) in
  for k = 1 to Array.length indeksi - 1 do
    let c = indeksi.(k) in
    match Hashtbl.find_opt slovar ((!predpona lsl 8) lor c) with
    | Some koda -> predpona := koda
    | None ->
        zapisi !predpona !velikost;
        if !naslednja >= 4095 then (
          zapisi pocisti !velikost;
          Hashtbl.reset slovar;
          naslednja := 258;
          velikost := 9)
        else (
          Hashtbl.add slovar ((!predpona lsl 8) lor c) !naslednja;
          incr naslednja;
          if !naslednja > 1 lsl !velikost && !velikost < 12 then incr velikost);
        predpona := c
  done;
  zapisi !predpona !velikost;
  zapisi konec !velikost;
  if !stevilo_bitov > 0 then Buffer.add_char izhod (Char.chr !zbiralnik);
  Buffer.contents izhod

let shrani_gif ime_datoteke slike =
  let visina = Array.length (List.hd slike) in
  let sirina = List.fold_left (fun m s -> max m (Array.length s.(0))) 0 slike in
  let bela = (255, 255, 255) in
  let paleta = Hashtbl.create 256 in
  let indeks ((r, g, m) as barva) =
    let kljuc = (r lsl 16) lor (g lsl 8) lor m in
    match Hashtbl.find_opt paleta kljuc with
    | Some (i, _) -> i
    | None ->
        let i = Hashtbl.length paleta in
        Hashtbl.add paleta kljuc (i, barva);
        i
  in
  let okvir slika =
    Array.init (visina * sirina) (fun k ->
        let vrstica = slika.(k / sirina) and j = k mod sirina in
        indeks (if j < Array.length vrstica then vrstica.(j) else bela))
  in
  let okvirji = List.map okvir slike in
  if Hashtbl.length paleta > 256 then invalid_arg "shrani_gif: več kot 256 barv";
  let barve = Array.make 256 bela in
  Hashtbl.iter (fun _ (i, barva) -> barve.(i) <- barva) paleta;
  let b = Buffer.create 1_000_000 in
  let bajt x = Buffer.add_char b (Char.chr (x land 0xFF)) in
  let dva_bajta x = bajt x; bajt (x lsr 8) in
  Buffer.add_string b "GIF89a";
  dva_bajta sirina; dva_bajta visina; bajt 0xF7; bajt 0; bajt 0;
  Array.iter (fun (r, g, m) -> bajt r; bajt g; bajt m) barve;
  Buffer.add_string b "\x21\xFF\x0BNETSCAPE2.0\x03\x01\x00\x00\x00";
  let zadnji = List.length okvirji - 1 in
  List.iteri
    (fun n indeksi ->
      Buffer.add_string b "\x21\xF9\x04\x00";
      dva_bajta (if n = zadnji then 300 else 4); bajt 0; bajt 0;
      bajt 0x2C; dva_bajta 0; dva_bajta 0; dva_bajta sirina; dva_bajta visina; bajt 0;
      bajt 8;
      let podatki = lzw indeksi in
      let rec bloki i =
        if i < String.length podatki then (
          let k = min 255 (String.length podatki - i) in
          bajt k;
          Buffer.add_substring b podatki i k;
          bloki (i + k))
      in
      bloki 0;
      bajt 0)
    okvirji;
  bajt 0x3B;
  Out_channel.with_open_bin ime_datoteke (fun kanal ->
      Out_channel.output_string kanal (Buffer.contents b))

In [ ]:
let skaliraj sirina slika =
    Array.map
      (fun vrstica ->
        let n = Array.length vrstica in
        Array.init sirina (fun j -> vrstica.(j * n / sirina)))
      slika

In [ ]:
let skaliraj_postopoma k slika =
    let n = Array.length slika.(0) in
    List.init (k + 1) (fun i -> skaliraj (n - i) slika)

In [ ]:
preberi_gif "veliki-val.gif" |> skaliraj_postopoma 148 |> shrani_gif "skaliranje.gif"

In [ ]:
let razdalja (r1, g1, b1) (r2, g2, b2) =
    abs (r1 - r2) + abs (g1 - g2) + abs (b1 - b2)

In [ ]:
let energije slika =
    Array.map
      (fun vrstica ->
        let n = Array.length vrstica in
        Array.init n (fun j ->
            let levo = if j = 0 then 1 else j - 1
            and desno = if j = n - 1 then n - 2 else j + 1 in
            razdalja vrstica.(levo) vrstica.(j)
            + razdalja vrstica.(j) vrstica.(desno)))
      slika

In [ ]:
let indeks_najmanjsega tabela =
    let rec aux najboljsi j =
      if j = Array.length tabela then najboljsi
      else aux (if tabela.(j) < tabela.(najboljsi) then j else najboljsi) (j + 1)
    in
    aux 0 1

In [ ]:
let siv_po_tockah energije = Array.map indeks_najmanjsega energije

In [ ]:
let navpicni_siv energije =
    let n = Array.length energije.(0) in
    let vsote =
      Array.init n (fun j ->
          Array.fold_left (fun s vrstica -> s + vrstica.(j)) 0 energije)
    in
    Array.make (Array.length energije) (indeks_najmanjsega vsote)

In [ ]:
let zavijajoci_siv energije =
    let m = Array.length energije and n = Array.length energije.(0) in
    let tabela = Array.make m [||] in
    let rec napolni i =
      if i >= 0 then (
        tabela.(i) <-
          Array.init n (fun j ->
              let e = energije.(i).(j) in
              if i = m - 1 then (e, j)
              else
                let sosed k =
                  if 0 <= k && k < n then (e + fst tabela.(i + 1).(k), k)
                  else (max_int, k)
                in
                min (sosed (j - 1)) (min (sosed j) (sosed (j + 1))));
        napolni (i - 1))
    in
    let rec sledi i j =
      if i = m then [] else j :: sledi (i + 1) (snd tabela.(i).(j))
    in
    napolni (m - 1);
    Array.of_list (sledi 0 (indeks_najmanjsega (Array.map fst tabela.(0))))

In [ ]:
let odstrani_siv slika siv =
    Array.mapi
      (fun i vrstica ->
        Array.init (Array.length vrstica - 1) (fun j ->
            if j < siv.(i) then vrstica.(j) else vrstica.(j + 1)))
      slika

In [ ]:
let pokazi_siv slika siv =
    Array.mapi
      (fun i vrstica ->
        Array.mapi (fun j barva -> if j = siv.(i) then (255, 0, 0) else barva) vrstica)
      slika

In [ ]:
let rec krci poisci_siv k slika =
    if k = 0 then [slika]
    else
      let siv = poisci_siv (energije slika) in
      pokazi_siv slika siv :: krci poisci_siv (k - 1) (odstrani_siv slika siv)

In [ ]:
preberi_gif "veliki-val.gif" |> krci siv_po_tockah 148 |> shrani_gif "po-tockah.gif"

In [ ]:
preberi_gif "veliki-val.gif" |> krci navpicni_siv 148 |> shrani_gif "navpicni.gif"

In [ ]:
preberi_gif "veliki-val.gif" |> krci zavijajoci_siv 148 |> shrani_gif "zavijajoci.gif"